# Legal Matter Intake Triage Multi-Agent System
### Built with Google ADK 2.0

---

## Project Architecture & Executive Summary

This notebook provides a complete, runnable demonstration of the **Matter Intake Triage Agent**. 
Legal intake is a complex process requiring entity extraction, deadline calculation, and strict compliance with privacy guidelines. This project utilizes the Google Agent Development Kit (ADK 2.0) to orchestrate a hierarchy of specialized AI agents that securely process legal intakes, calculate deadlines, redact Personally Identifiable Information (PII), and synthesize a final structured intake packet.

### Core Agent Architecture

The system uses a stateful orchestration framework where a **Coordinator Agent** delegates tasks to specialized sub-agents:

1. **Security & Compliance Agent**: Intercepts the raw intake, redacts PII (emails, phones, SSNs), blocks requests for direct legal advice, and logs actions.
2. **Intake Classifier Agent**: Determines the overall matter type (e.g., Litigation, Corporate) and urgency level.
3. **Document Extraction Agent**: Parses the sanitized text to extract involved parties, dates, locations, referenced documents, and legal claims.
4. **Deadline Triage Agent**: Uses specialized date-calculation tools to identify intervals and generates mock calendar events for statute of limitations.
5. **Packet Writer Agent**: Aggregates all findings into a structured Intake Memo and creates a checklist for missing information.


# Section 1 - Environment Setup
In a Kaggle environment, we first need to ensure the required libraries are installed and our API keys are configured.

In [ ]:
# Install dependencies (uncomment if running in a fresh Kaggle kernel)
# !pip install google-adk>=2.0.0 pydantic pytest

import os
from google.colab import userdata

# If running in Kaggle, you would use Kaggle Secrets. 
# For this notebook, we assume GOOGLE_API_KEY is set in the environment.
# os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
print("Environment initialized.")

# Section 2 - Defining the Tool Layer
Before agents can operate, they need deterministic tools to execute specific tasks. We define tools for PII redaction, date extraction, date calculation, mock calendar events, policy lookup, and audit logging.

In [ ]:
import re
from datetime import datetime
from google.adk.tools import ToolContext

# Tool 1: Date Tools
def calculate_days_between_dates(date_str1: str, date_str2: str) -> dict:
    """Calculates the number of days between two dates (YYYY-MM-DD)."""
    try:
        d1 = datetime.strptime(date_str1, "%Y-%m-%d")
        d2 = datetime.strptime(date_str2, "%Y-%m-%d")
        return {"status": "success", "days": abs((d2 - d1).days)}
    except ValueError as e:
        return {"status": "error", "message": str(e)}

def extract_dates_regex(text: str) -> dict:
    """Extracts date strings from text."""
    dates = re.findall(r'\b(\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4})\b', text)
    return {"status": "success", "dates": dates}

# Tool 2: Redaction Tool
def redact_pii(text: str) -> dict:
    """Redacts PII from text."""
    text = re.sub(r'[\w\.-]+@[\w\.-]+', '[EMAIL REDACTED]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE REDACTED]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN REDACTED]', text)
    return {"status": "success", "redacted_text": text}

# Tool 3: Mock Calendar
def create_mock_calendar_event(title: str, date: str, description: str) -> dict:
    """Creates a mock calendar event for deadline triage."""
    event_id = f"evt_{hash(title + date) % 100000}"
    return {"status": "success", "event": {"id": event_id, "title": title, "date": date}}

# Tool 4: Policy Lookup
def lookup_mock_matter_type_policy(matter_type: str) -> dict:
    """Looks up policy guidelines for a specific legal matter type."""
    policies = {
        "Litigation": "Requires immediate deadline triage and preservation notices.",
        "Corporate": "Standard 3-day SLA for document extraction."
    }
    return {"status": "success", "policy": policies.get(matter_type, "Standard policy applies.")}

# Tool 5: Audit Logging
def write_audit_log(action: str, details: str, tool_context: ToolContext) -> dict:
    """Writes an audit log entry for security and compliance tracking."""
    if "audit_logs" not in tool_context.state:
        tool_context.state["audit_logs"] = []
    tool_context.state["audit_logs"].append({"action": action, "details": details})
    return {"status": "success", "message": "Audit log recorded successfully."}

print("Tools initialized.")

# Section 3 - Multi-Agent Architecture
We now define our ADK 2.0 specialized agents using Pydantic schemas to enforce structured outputs.

In [ ]:
from pydantic import BaseModel, Field
from google.adk.agents import Agent

# Agent 1: Security Reviewer
class SecurityReviewOutput(BaseModel):
    redacted_text: str
    legal_advice_requested: bool
    compliance_flags: list[str]

security_reviewer = Agent(
    name="security_reviewer",
    model="gemini-1.5-flash",
    mode="task",
    tools=[redact_pii, write_audit_log],
    output_schema=SecurityReviewOutput,
    description="Redacts PII, blocks legal advice requests, and writes audit logs.",
    instruction="""You are the Security and Compliance Reviewer.
    1. Check if the input asks for legal advice. If so, set legal_advice_requested to true.
    2. Use the redact_pii tool to clean the input text of personal information.
    3. Write an audit log using the write_audit_log tool noting the redaction.
    Call finish_task with your results when done."""
)

# Agent 2: Intake Classifier
class ClassificationOutput(BaseModel):
    matter_type: str
    urgency: str
    summary: str

intake_classifier = Agent(
    name="intake_classifier",
    model="gemini-1.5-flash",
    mode="task",
    output_schema=ClassificationOutput,
    description="Classifies the matter type and urgency of an intake email or document.",
    instruction="You are an expert legal intake classifier. Analyze the provided text to determine the matter type and its urgency."
)

# Agent 3: Document Extractor
class DocumentExtractionOutput(BaseModel):
    parties: list[str]
    dates: list[str]
    locations: list[str]
    documents_referenced: list[str]
    claims: list[str]

document_extractor = Agent(
    name="document_extractor",
    model="gemini-1.5-flash",
    mode="task",
    output_schema=DocumentExtractionOutput,
    description="Extracts key entities like parties, dates, locations, documents, and claims.",
    instruction="You are a precise legal document extraction assistant. Analyze the text and extract the required information."
)

# Agent 4: Deadline Triage
class DeadlineTriageOutput(BaseModel):
    calculated_deadlines: list[str]
    uncertainty_flag: bool
    calendar_events_created: list[str]

deadline_triage = Agent(
    name="deadline_triage",
    model="gemini-1.5-flash",
    mode="task",
    tools=[calculate_days_between_dates, extract_dates_regex, create_mock_calendar_event],
    output_schema=DeadlineTriageOutput,
    description="Calculates date intervals for deadlines, flags uncertainty, and creates calendar events.",
    instruction="You calculate deadlines based on dates. Use tools to extract dates and create calendar events if deadlines are identified."
)

# Agent 5: Packet Writer
class PacketOutput(BaseModel):
    intake_memo: str
    missing_info_checklist: list[str]

packet_writer = Agent(
    name="packet_writer",
    model="gemini-1.5-flash",
    mode="task",
    tools=[lookup_mock_matter_type_policy],
    output_schema=PacketOutput,
    description="Generates the final structured intake memo and missing info checklist.",
    instruction="Synthesize outputs from the classifier, extractor, deadline triage, and security review. Use the policy tool to get guidelines and output a final memo."
)

# Central Coordinator Agent
coordinator_agent = Agent(
    name="coordinator",
    model="gemini-1.5-flash",
    description="The main coordinator agent that delegates tasks to handle legal matter intake.",
    instruction="""You are the Coordinator Agent for Legal Matter Intake.
    Process incoming legal matters by delegating to your sub-agents in this order:
    1. security_reviewer (to redact PII)
    2. intake_classifier (to classify)
    3. document_extractor (to extract entities)
    4. deadline_triage (to handle dates)
    5. packet_writer (to synthesize the final memo)
    """,
    sub_agents=[security_reviewer, intake_classifier, document_extractor, deadline_triage, packet_writer]
)
print("Agents initialized.")

# Section 4 - Orchestration Router & Runtime Execution
We provide a sample raw intake email. The `Runner` class will initialize the state and process the intake through the coordinator agent.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import asyncio

sample_intake_email = """
From: client@example.com
To: intake@lawfirm.com
Subject: New slip and fall case at the mall

Hi, 
My name is John Doe (email: john.doe@email.com, phone: 555-987-6543, SSN: 987-65-4321). I was at the Mega Mall on 2026-05-15 and slipped on a wet floor near the food court. I want to sue them for damages. I also have the incident report from mall security. Can you help me with this and give me some legal advice on what to do next? The statute of limitations might be an issue.
"""

async def run_intake_system():
    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name="legal_triage", user_id="test_user", session_id="sess_001")
    runner = Runner(agent=coordinator_agent, app_name="legal_triage", session_service=session_service)
    
    print("Starting Multi-Agent processing...\n")
    async for event in runner.run_async(
        user_id="test_user", 
        session_id="sess_001",
        new_message=types.Content(role="user", parts=[types.Part.from_text(text=sample_intake_email)]),
    ):
        if event.is_final_response():
            print("\n======================= FINAL INTAKE PACKET =======================")
            print(event.content.parts[0].text)

# Note: In Jupyter environments, an event loop is already running. 
# We can use `await` directly in the cell, or use asyncio.run if outside.
import nest_asyncio
nest_asyncio.apply()

try:
    asyncio.run(run_intake_system())
except Exception as e:
    print(f"Execution requires valid GOOGLE_API_KEY. Error: {e}")

## Conclusion
This framework successfully demonstrates how complex workflows involving privacy checks, entity extraction, calculations, and synthesis can be orchestrated securely and deterministically using a multi-agent system.